# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Clinicopathological CRC Survivors dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

**Dataset source:** Provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant Schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load metadata and dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List all available record sets, their fields (columns), and their Croissant `@id`s for reference.

In [ ]:
# The mlcroissant API provides an interface to list record sets and fields.
from pprint import pprint

print("Record sets available in the dataset:\n")
for record_set in dataset.record_sets:
    print(f"Record Set: {record_set['@id']} | Name: {record_set.get('name','(no name)')}")
    if 'field' in record_set:
        print("  Fields:")
        # Each field is a dict with '@id' and usually 'name'
        for field in record_set['field']:
            # Some datasets have field as dict, some as list
            if isinstance(field, dict):
                print(f"    - {field.get('@id')} | Name: {field.get('name','(no name)')}")
            else:
                # If only @id provided
                print(f"    - {field}")
    print()
if not dataset.record_sets:
    print("No explicit record sets found in top-level schema. Attempting to automatically infer.")

# For many real-world schemas, record_sets might be empty and data must be loaded from concrete file objects. See next cell.

## 3. Data Extraction
Load all record sets as DataFrames using their `@id`. If no explicit record sets are defined, attempt to infer from data files.

In [ ]:
# Attempt to enumerate available record set IDs (use top-level schema if recordSet is empty)
record_set_ids = []
# Try loading by record_set if present
if dataset.record_sets:
    record_set_ids = [rs['@id'] for rs in dataset.record_sets]
else:
    # If recordSet is empty, mlcroissant's records() can still fetch records from data files
    # We'll call .records() with no arguments to get all top-level records
    print("No explicit record sets found. Loading available records from first file object.")

dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set '{record_set_id}' with columns: {df.columns.tolist()}")
    # For demo, pick the first record set for further exploration
    main_record_set_id = record_set_ids[0]
else:
    # Load raw records (should be a single main tabular file)
    records = list(dataset.records())
    df = pd.DataFrame(records)
    main_record_set_id = 'main'  # Artificial ID
    dataframes[main_record_set_id] = df
    print(f"Loaded {len(df)} records from main data file with columns: {df.columns.tolist()}")

# Show top of the loaded DataFrame
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Process the main dataset by filtering numeric fields and performing group analysis. 

Choose a relevant numeric field (`@id`) for the analysis. Here, we attempt to guess typical field names (e.g. 'Age', 'Interval', 'Metastases'), falling back to any numeric-looking column.

In [ ]:
df = dataframes[main_record_set_id]
# Inspect columns to find numeric fields (by common name/heuristic or dtype, as the @id is the DataFrame column name)
numeric_candidates = [col for col in df.columns if ('age' in col.lower() or 'interval' in col.lower() or 'number' in col.lower() or 'metast' in col.lower())]
if not numeric_candidates:
    # If not found by name, fall back to columns with numeric dtype
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()

if not numeric_candidates:
    print("No apparent numeric columns found.\nColumns:", df.columns.tolist())
else:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for EDA: '{numeric_field}'")

    # Convert to numeric if necessary (handle NaN, missing values)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter records with value > threshold (e.g., filter age > 45 or interval > 10 months)
    threshold = df[numeric_field].quantile(0.5)  # Use median as demonstration threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.1f} (median): {len(filtered_df)} rows")

    # Normalize this field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field, e.g. sex, anatomical site, msi status etc.
    group_field_candidates = [
        col for col in df.columns if ('sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower() or 'site' in col.lower())
    ]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
        print(f"\nGrouped mean of {numeric_field} by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No apparent grouping field found for EDA.")

## 5. Visualization
Display distributions and relationships between fields. Here, we plot the numeric field's distribution and, if possible, by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_candidates:
    print("No numeric column determined for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    # If allowed by group
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion

In this notebook, we used the FAIR² CRC Survivors dataset's Croissant schema via `mlcroissant` to explore metadata, load and analyze records, filter and normalize key numeric fields (e.g., age, interval), and visualize the results. The use of Croissant `@id`s makes code robust and portable.

Key observations:
- The dataset provides detailed case-level clinical and molecular attributes for second primary colorectal cancer in survivors.
- EDA and visualization steps (e.g., filtering by age or clinical intervals) can help reveal cohort patterns or group differences (such as MSI status or anatomical location).
- `mlcroissant` makes FAIR data integration straightforward and reproducible for machine learning and research workflows.